# C2-linear-models — Session 2: Normal Equations, Rank, and the Pseudoinverse

*One class session, roughly 90 minutes. Builds on Session 1's model,
residual, MSE, gradient, and orthogonality pictures.*

This session turns the zero-gradient condition into the normal
equations, identifies exactly when coefficients are unique, separates
an inconsistent data equation from least squares, and uses the
pseudoinverse to select a minimum-norm solution at deficient rank.

Try each checkpoint before the collected answers at the end. All
numerical comparisons state \`atol\` and \`rtol\` explicitly.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

## 1. One Design-Matrix Convention

Session 1 used
$\hat y_i=\sum_{k=1}^{d}X_{ik}w_k+b$.
For closed-form algebra, prepend a ones column and put the intercept
first:
\[
\widetilde X=[\mathbf 1\ \ X]\in\mathbb R^{n\times p},
\qquad
\beta=(b,w_1,\ldots,w_d)^T\in\mathbb R^p.
\]
We now shorten $\widetilde X$ to $X$:
\[
X:(n,p),\quad \beta:(p,),\quad y:(n,),\quad
\hat y=X\beta:(n,),\quad r=X\beta-y:(n,).
\]
Never both prepend ones and add a separate bias.

For $L(\beta)=\lVert X\beta-y\rVert_2^2/n$,
\[
\nabla_\beta L=\frac{2}{n}X^T(X\beta-y).
\]
The $2/n$ sets the gradient scale but cancels at zero.


### From MSE to the normal equations

At a minimizer $\hat\beta$, zero gradient gives
\[
\frac{2}{n}X^T(X\hat\beta-y)=0
\quad\Longleftrightarrow\quad
\boxed{X^TX\hat\beta=X^Ty}.
\]
Equivalently, $X^T\hat r=0$ for
$\hat r=X\hat\beta-y$: the residual is orthogonal to every column.

This implication runs backward. If $X^T\hat r=0$ and
$h=\beta-\hat\beta$, then
\[
\lVert X\beta-y\rVert^2
=\lVert\hat r+Xh\rVert^2
=\lVert\hat r\rVert^2+\lVert Xh\rVert^2.
\]
The cross term vanishes, so every normal-equation solution is a global
minimizer.

### Checkpoint 1

1. A raw feature array has shape `(60, 4)` and needs an intercept.
   Give the augmented shapes of $X$, $\beta$, $X^Ty$, and $X^TX$.
2. Starting from $L=\lVert X\beta-y\rVert^2/n$, derive the gradient
   and explain why its zero condition means residual orthogonality.

## 2. Full Column Rank and a Unique Worked Fit

Full column rank means $\operatorname{rank}(X)=p$. For every nonzero
$v\in\mathbb R^p$,
\[
v^TX^TXv=\lVert Xv\rVert_2^2>0.
\]
Thus $X^TX$ is positive definite and invertible. Conversely, dependent
columns give a nonzero $v$ with $Xv=0$, so $X^TXv=0$. Hence
\[
X\text{ full column rank}
\Longleftrightarrow X^TX\text{ positive definite}
\Longleftrightarrow X^TX\text{ invertible}.
\]
The inequality $n\ge p$ is necessary, not sufficient: even $n>p$
allows duplicate/dependent columns.

When full rank is guaranteed, use
\`np.linalg.solve(X.T @ X, X.T @ y)\`.
Do not form \`np.linalg.inv(X.T @ X)\`.


### Worked example: a unique fit

With a ones column,
\[
X=\begin{pmatrix}1&0\\1&1\\1&3\end{pmatrix},\qquad
y=\begin{pmatrix}1\\2\\2\end{pmatrix}.
\]
Then
\[
X^TX=\begin{pmatrix}3&4\\4&10\end{pmatrix},\qquad
X^Ty=\begin{pmatrix}5\\8\end{pmatrix}.
\]
The determinant is $14>0$, and solving gives
$\hat\beta=(9/7,2/7)^T$. The residual is nonzero because the points are
not exactly collinear, but $X^T\hat r=0$.

### Checkpoint 2

1. Prove $X$ full column rank iff $X^TX$ is positive definite and
   invertible.
2. Give a dependent $3\times2$ matrix with $n>p$, a nonzero null
   vector, and explain why row count is insufficient.

In [ ]:
X_full = np.array([[1.0, 0.0],
                   [1.0, 1.0],
                   [1.0, 3.0]])
y_full = np.array([1.0, 2.0, 2.0])

gram_full = X_full.T @ X_full
rhs_full = X_full.T @ y_full
beta_full = np.linalg.solve(gram_full, rhs_full)
resid_full = X_full @ beta_full - y_full

assert np.allclose(beta_full, np.array([9 / 7, 2 / 7]),
                   atol=ATOL, rtol=RTOL)
assert np.allclose(X_full.T @ resid_full, np.zeros(2),
                   atol=ATOL, rtol=RTOL)

## 3. Consistency, Rank Deficiency, and Identifiability

Two questions differ:

1. Does $X\beta=y$ have an exact solution? This requires
   $y\in\operatorname{col}(X)$.
2. Which $\beta$ minimizes $\lVert X\beta-y\rVert^2$?

A least-squares minimizer always exists in finite dimensions. The
normal equations are consistent for every $X,y$, even when $X^TX$ is
singular. Singularity may destroy coefficient uniqueness, not existence
of a closest fitted vector. Every minimizer produces the same projection
of $y$ onto $\operatorname{col}(X)$.


### The nullspace family

If $z\ne0$ lies in $\mathcal N(X)$, then
$X(\beta+z)=X\beta$. The data cannot distinguish those coefficients.
If $\beta_0$ solves the normal equations, every minimizer is exactly
\[
\boxed{\beta_0+z,\qquad z\in\mathcal N(X)}.
\]
Indeed, a second minimizer has loss difference
$\lVert X(\beta-\beta_0)\rVert^2=0$, so its difference is in the
nullspace. Thus coefficients can be non-identifiable while the fitted
vector is unique.

### Checkpoint 3

1. If $X\beta=y$ is inconsistent, why do least-squares minimizers
   still exist, and what is their residual perpendicular to?
2. Prove that all minimizers are $\beta_0+\mathcal N(X)$ and explain
   why their predictions remain unique.

## 4. Pseudoinverse and the Minimum-Norm Worked Fit

The pseudoinverse exists for every rectangular matrix. Define
\[
\beta^+=X^+y.
\]
It is a least-squares minimizer and the unique one with smallest
Euclidean norm. NumPy's route is \`np.linalg.pinv(X) @ y\`.

At full column rank,
$X^+=(X^TX)^{-1}X^T$, so this agrees mathematically with the normal
system. This identity does not make rectangular $X$ ordinarily
invertible and does not justify blindly forming a Gram inverse.

Why minimum norm? $\beta^+$ lies in the row space, orthogonal to
$\mathcal N(X)$. Every minimizer is $\beta^++z$, so
\[
\lVert\beta^++z\rVert^2
=\lVert\beta^+\rVert^2+\lVert z\rVert^2
\ge\lVert\beta^+\rVert^2.
\]

### Worked example: one fit, many coefficients

Let
\[
X=\begin{pmatrix}
1&1&2\\1&2&4\\1&3&6\\1&4&8
\end{pmatrix},\qquad
y=\begin{pmatrix}2\\2.5\\4.1\\5\end{pmatrix}.
\]
Column three is twice column two, so
$z=(0,-2,1)^T\in\mathcal N(X)$. For any scalar $t$,
$\beta^++tz$ has the same predictions and residual as $\beta^+$.
Orthogonality makes its norm smallest at $t=0$.

### Checkpoint 4

1. State the pseudoinverse least-squares and minimum-norm contracts,
   and explain why they agree with the normal system at full rank.
2. For $z\in\mathcal N(X)$, expand
   $\lVert\beta^++z\rVert^2$ and locate its minimum.

In [ ]:
X_def = np.array([[1.0, 1.0, 2.0],
                  [1.0, 2.0, 4.0],
                  [1.0, 3.0, 6.0],
                  [1.0, 4.0, 8.0]])
y_def = np.array([2.0, 2.5, 4.1, 5.0])
z_def = np.array([0.0, -2.0, 1.0])

beta_plus = np.linalg.pinv(X_def) @ y_def
family_t = np.array([-3.0, -0.5, 0.0, 1.25, 4.0])
beta_family = beta_plus[None, :] + family_t[:, None] * z_def
pred_family = X_def @ beta_family.T
norm_family = np.linalg.norm(beta_family, axis=1)

assert np.allclose(X_def @ z_def, np.zeros(4), atol=ATOL, rtol=RTOL)
assert np.allclose(pred_family,
                   np.repeat((X_def @ beta_plus)[:, None], len(family_t), axis=1),
                   atol=ATOL, rtol=RTOL)
assert int(np.argmin(norm_family)) == 2
assert np.allclose(X_def.T @ (X_def @ beta_plus - y_def),
                   np.zeros(3), atol=ATOL, rtol=RTOL)

## 5. Projectors Reveal What Is Unique

\[
P_{\mathrm{col}}=XX^+,\qquad P_{\mathrm{row}}=X^+X.
\]

- $P_{\mathrm{col}}y=X\beta^+$ is the unique fitted vector.
- With our convention,
  $r=X\beta^+-y=-(I-P_{\mathrm{col}})y$.
- $P_{\mathrm{row}}\beta$ keeps the identifiable row-space part.
- $(I-P_{\mathrm{row}})\beta\in\mathcal N(X)$ cannot affect predictions.

Orthogonal projectors satisfy $P^T=P$ and $P^2=P$. Check these and
orthogonality with tolerances, not exact float equality.

### Checkpoint 5

1. Name the spaces projected onto by $XX^+$ and $X^+X$.
2. Show why $(I-X^+X)\beta$ cannot change predictions, and state the
   symmetry/idempotence identities of an orthogonal projector.

In [ ]:
P_col = X_def @ np.linalg.pinv(X_def)
P_row = np.linalg.pinv(X_def) @ X_def
I_row = np.eye(X_def.shape[1])

assert np.allclose(P_col.T, P_col, atol=ATOL, rtol=RTOL)
assert np.allclose(P_col @ P_col, P_col, atol=ATOL, rtol=RTOL)
assert np.allclose(P_row.T, P_row, atol=ATOL, rtol=RTOL)
assert np.allclose(P_row @ P_row, P_row, atol=ATOL, rtol=RTOL)
assert np.allclose(X_def @ ((I_row - P_row) @ np.array([2.0, -1.0, 3.0])),
                   np.zeros(X_def.shape[0]), atol=ATOL, rtol=RTOL)

## 6. Numerical Rank and Tolerance Discipline

Exact algebra has zero singular values; floating-point data may have
tiny ones. Rank is therefore scale-dependent numerically.

- Use \`np.linalg.solve\` only under a full-rank contract.
- Use \`np.linalg.pinv\` for general-rank minimum-norm least squares.
- Inspect singular values or condition number near dependence.
- State \`atol\` and \`rtol\`; do not test float identities with \`==\`.

A Gram matrix can be invertible on paper yet dangerously ill
conditioned.

### Checkpoint 6

1. Why can an algebraically invertible Gram matrix still be unsafe for
   the normal-equation solve?
2. Write an explicit `np.allclose` residual-orthogonality check with
   both `atol` and `rtol`.

In [ ]:
eps = 1e-8
X_near = np.array([[1.0, 1.0],
                   [2.0, 2.0 + eps],
                   [3.0, 3.0 - eps],
                   [4.0, 4.0 + 2 * eps]])
singular_values = np.linalg.svd(X_near, compute_uv=False)
condition_number = np.linalg.cond(X_near)

assert singular_values[0] > singular_values[1] > 0.0
assert condition_number > 1e8

## 7. Common Pitfalls

1. Prepending ones and also adding a bias.
2. Losing the $2/n$ because another source used $1/(2n)$ loss.
3. Claiming $n>p$ makes columns independent.
4. Claiming inconsistency of $X\beta=y$ prevents least squares.
5. Treating $X^+$ as an ordinary inverse or inverting singular $X^TX$.
6. Confusing non-unique coefficients with non-unique predictions.
7. Using exact equality for floating-point identities.

### Checkpoint 7

1. Diagnose the error in “$n>p$, so the coefficients are identifiable.”
2. For deficient rank, state separately what is and is not unique, and
   why $X^+$ must not be described as an ordinary rectangular inverse.

## Exam Connections

Proof questions move among zero gradient, normal equations, residual
orthogonality, and projection. Write dimensions before multiplying.

Coding contracts pin one route:

- **full rank:** form the Gram system and call \`np.linalg.solve\`;
- **general rank:** call \`np.linalg.pinv(X) @ y\` and verify fitted,
  residual, and minimum-norm properties.

API bans are literal: \`inv\`, \`pinv\`, \`lstsq\`, \`solve\`, and a
library estimator are not interchangeable because they agree on one
friendly example.

## Going Deeper

F6 explains the pseudoinverse through the SVD. C3 supplies iterative
optimization for large problems and objectives changed by penalties.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $X:(60,5)$, $\beta:(5,)$, $X^Ty:(5,)$, $X^TX:(5,5)$.
2. $\nabla L=(2/n)X^T(X\beta-y)$; at zero,
   $X^Tr=0$, so $r\perp\operatorname{col}(X)$.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $v^TX^TXv=\lVert Xv\rVert^2$ is positive for every nonzero $v$
   exactly when $X$ has trivial nullspace; for symmetric Gram matrices,
   positive definiteness is equivalent to invertibility.
2. Example $X=\begin{pmatrix}1&2\\2&4\\3&6\end{pmatrix}$,
   $v=(-2,1)^T$. More rows do not prevent dependent columns.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Projection onto the finite-dimensional column space always exists;
   the residual is perpendicular to that space.
2. The loss difference is
   $\lVert X(\beta-\beta_0)\rVert^2$, so equality requires
   $\beta-\beta_0\in\mathcal N(X)$; null shifts vanish under $X$.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $\beta^+=X^+y$ minimizes residual norm and then coefficient norm.
   At full rank it equals the unique normal-system solution.
2. Row space is orthogonal to nullspace, so
   $\lVert\beta^++z\rVert^2=\lVert\beta^+\rVert^2+\lVert z\rVert^2$,
   minimized uniquely at $z=0$.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. $XX^+$ projects onto the column space; $X^+X$ onto the row space.
2. $(I-X^+X)\beta\in\mathcal N(X)$; an orthogonal projector satisfies
   $P^T=P$ and $P^2=P$.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Near-dependent columns make the Gram condition number large, so
   roundoff is amplified despite exact invertibility.
2. `np.allclose(X.T @ (X @ beta - y), 0.0, atol=ATOL, rtol=RTOL)`.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. $n>p$ is only necessary room for independence; it does not prove it.
2. Coefficients are non-unique, while the fitted vector/residual are
   unique. $X^+$ is a general Moore–Penrose pseudoinverse, not an
   ordinary inverse of rectangular $X$.

</details>